In [39]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [40]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
48,Mild SPOILERS contained herein. I'm spoiling t...,negative
459,I believe an entire book can be written about ...,negative
60,What could've been a great film about the late...,negative
802,I'll be honest-- the pimped out purple plane w...,negative
861,This movie received a great write up in Blockb...,negative


In [41]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [42]:
df = normalize_text(df)
df.head()

,review,sentiment
48,mild spoiler contained herein spoiling film sa...,negative
459,believe entire book written odyssey remake cla...,negative
60,could ve great film late poker pro pre poker c...,negative
802,honest pimped purple plane snoop dogg helm amu...,negative
861,movie received great write blockbuster coming ...,negative


In [43]:
df['sentiment'].value_counts()

sentiment
negative    266
positive    234
Name: count, dtype: int64

In [44]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [45]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
48,mild spoiler contained herein spoiling film sa...,0
459,believe entire book written odyssey remake cla...,0
60,could ve great film late poker pro pre poker c...,0
802,honest pimped purple plane snoop dogg helm amu...,0
861,movie received great write blockbuster coming ...,0


In [46]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [47]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/sKyi01/Capstone-MLOps-Project.mlflow')
dagshub.init(repo_owner='sKyi01', repo_name='Capstone-MLOps-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-03-06 22:46:00,462 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/sKyi01/Capstone-MLOps-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "sKyi01/Capstone-MLOps-Project"

2026-03-06 22:46:00,513 - INFO - Initialized MLflow to track repo "sKyi01/Capstone-MLOps-Project"


Repository sKyi01/Capstone-MLOps-Project initialized!

2026-03-06 22:46:00,527 - INFO - Repository sKyi01/Capstone-MLOps-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/c23d43bc492d4624bca22492563d962c', creation_time=1772816935392, experiment_id='0', last_update_time=1772816935392, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [50]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-03-06 22:46:01,040 - INFO - Starting MLflow run...
2026-03-06 22:46:01,600 - INFO - Logging preprocessing parameters...
2026-03-06 22:46:02,615 - INFO - Initializing Logistic Regression model...
2026-03-06 22:46:02,616 - INFO - Fitting the model...
2026-03-06 22:46:02,680 - INFO - Model training complete.
2026-03-06 22:46:02,681 - INFO - Logging model parameters...
2026-03-06 22:46:03,007 - INFO - Making predictions...
2026-03-06 22:46:03,014 - INFO - Calculating evaluation metrics...
2026-03-06 22:46:03,069 - INFO - Logging evaluation metrics...
2026-03-06 22:46:04,478 - INFO - Saving and logging the model...
2026/03/06 22:46:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/06 22:46:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run polite-colt-702 at: https://dagshub.com/sKyi01/Capstone-MLOps-Project.mlflow/#/experiments/0/runs/29f1a702dacd4bd299a1022384063443
🧪 View experiment at: https://dagshub.com/sKyi01/Capstone-MLOps-Project.mlflow/#/experiments/0
